# 01 · Exploração da Olist

Monta a tabela de comentários (avaliações × pedidos × itens × clientes) e reproduz os números do pré-projeto. A Olist é o **domínio de aplicação**: aqui não há rótulo de aspecto, e a verdade de campo é o atraso real da entrega.

In [ ]:
# Configuração: no Colab, clona o repositório, instala o pacote `absa`
# e guarda cache do LLM, modelos e predições no Drive (sobrevive à queda da sessão)
import sys
if "google.colab" in sys.modules:
    !git clone -q https://github.com/vnevesl/pln-absa-olist.git 2>/dev/null || git -C pln-absa-olist pull -q
    %cd pln-absa-olist
    !pip -q install -e "."
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p /content/drive/MyDrive/pln-absa-olist/llm_cache /content/drive/MyDrive/pln-absa-olist/models /content/drive/MyDrive/pln-absa-olist/interim
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/llm_cache data/llm_cache
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/models models
    !ln -sfn /content/drive/MyDrive/pln-absa-olist/interim data/interim

In [ ]:
import pandas as pd
from absa.data import load_reviews

df = load_reviews()
print(df.shape)
df.head()

## Volume e notas

In [ ]:
print("comentários:", len(df), "| textos distintos:", df.texto.nunique())
print(df.nota.value_counts(normalize=True).sort_index().mul(100).round(1))
df.n_palavras.describe(percentiles=[.5, .95]).round(1)

## Atraso × nota (base da validação extrínseca)

In [ ]:
entregues = df[df.entregue]
entregues.groupby("atrasado").agg(
    comentarios=("review_id", "size"),
    nota_media=("nota", "mean"),
    pct_nota_baixa=("nota", lambda s: 100 * (s <= 2).mean()),
).round(2)

In [ ]:
faixas = pd.cut(entregues.atraso_dias, [-60, -10, -1, 0, 3, 7, 15, 200])
entregues.groupby(faixas, observed=True).nota.agg(lambda s: 100 * (s <= 2).mean()).round(1)

## Região e categoria

In [ ]:
print(df.regiao.value_counts())
df.categoria.value_counts().head(15)

In [ ]:
for _, r in df.sample(15, random_state=1).iterrows():
    print(f"[{r.nota}★ | atrasado={r.atrasado}] {r.texto}")